In [ ]:
import zipfile


# Unzip all contents into the current directory
with zipfile.ZipFile("ARR_215_resistor_photos.zip", 'r') as zip_ref:
    zip_ref.extractall('unzipped_photos')


In [ ]:
!ls "unzipped_photos"

In [ ]:
main_base_image_dir = 'unzipped_photos/ARR_215_resistor_photos'
# Target color and tolerance as used in the last successful run
target_rgb = (88, 77, 55)
tolerance_r = 9
tolerance_g = 12
tolerance_b = 8

# Define the color range for each channel
min_r, max_r = target_rgb[0] - tolerance_r, target_rgb[0] + tolerance_r
min_g, max_g = target_rgb[1] - tolerance_g, target_rgb[1] + tolerance_g
min_b, max_b = target_rgb[2] - tolerance_b, target_rgb[2] + tolerance_b

MIN_BRIGHTNESS = 125  # R+G+B sum — cuts deep shadows, lowest real dark brown ~136
PROCESS_SCALE = 1

print("hello world")

In [ ]:
import psutil

def print_mem():
    mem = psutil.virtual_memory()
    print(f"RAM: {mem.used/1e9:.1f}GB used / {mem.total/1e9:.1f}GB total ({mem.percent}%)")

print_mem()  # baseline before loop starts

In [ ]:
import builtins
builtins_min = builtins.min
builtins_max = builtins.max
from scipy.ndimage import uniform_filter
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# --- Valid resistor color ranges ---
color_ranges = [
    (55,  99,  46, 77,  33, 70),   # dark brown (r_min up 29→55, g_min up 23→46)
    (81, 122,  68, 115, 58, 99),   # mid brown
    (111, 183, 100, 152, 81, 125), # light brown
]
WARMTH_RATIO   = 1.3
MAX_AREA       = 0.0211e8
MIN_DIMENSION  = 150
MIN_BRIGHTNESS = 125

# --- Density check parameters ---
DENSITY_KERNEL = 15
DENSITY_THRESH = 0.48

all_cropped_images_filtered = []
rejected_no_color = []
rejected_too_large = []
processed_image_count = 0
padded_count = 0

def build_combined_mask(img_array):
    R = img_array[:, :, 0]
    G = img_array[:, :, 1]
    B = img_array[:, :, 2]

    safe_B = np.where(B > 0, B, 1)
    warmth_mask = (R / safe_B > WARMTH_RATIO) & (B > 0)
    brightness_mask = (R + G + B) >= MIN_BRIGHTNESS

    color_mask = np.zeros(R.shape, dtype=bool)
    for (r0, r1, g0, g1, b0, b1) in color_ranges:
        color_mask |= (
            (R >= r0) & (R <= r1) &
            (G >= g0) & (G <= g1) &
            (B >= b0) & (B <= b1)
        )

    return color_mask & warmth_mask & brightness_mask

def apply_density_filter(mask, kernel, threshold):
    """Keep only pixels where >= threshold fraction of kernel neighbourhood are also brown."""
    density = uniform_filter(mask.astype(float), size=kernel, mode='constant', cval=0)
    return mask & (density >= threshold)

for folder_root, sub_dirs, files in os.walk(main_base_image_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(folder_root, file)
            processed_image_count += 1
            print_mem()
            print(f"{100 * (processed_image_count-1)/420:.1f}% completion")
            print(f"too large: {len(rejected_too_large)}")
            print(f"no match: {len(rejected_no_color)}")
            try:
                with Image.open(image_path) as img:
                    img_rgb = img.convert('RGB')
                    img_array = np.array(img_rgb).astype(float)

                    combined_mask = build_combined_mask(img_array)
                    dense_mask    = apply_density_filter(combined_mask, DENSITY_KERNEL, DENSITY_THRESH)
                    matching_coords = np.argwhere(dense_mask)

                    if matching_coords.size == 0:
                        print("no color match")
                        rejected_no_color.append({
                            'filename': os.path.basename(file),
                            'resistor_value_folder': os.path.basename(folder_root),
                            'filepath': image_path
                        })
                        continue

                    min_y, min_x = matching_coords.min(axis=0)
                    max_y, max_x = matching_coords.max(axis=0)
                    img_h, img_w = img_array.shape[:2]

                    if (max_x - min_x + 1) < MIN_DIMENSION:
                        pad = (MIN_DIMENSION - (max_x - min_x + 1)) // 2
                        min_x = builtins_max(0, min_x - pad)
                        max_x = builtins_min(img_w - 1, max_x + pad)
                        if (max_x - min_x + 1) < MIN_DIMENSION:
                            min_x = builtins_max(0, max_x - MIN_DIMENSION + 1)

                    if (max_y - min_y + 1) < MIN_DIMENSION:
                        pad = (MIN_DIMENSION - (max_y - min_y + 1)) // 2
                        min_y = builtins_max(0, min_y - pad)
                        max_y = builtins_min(img_h - 1, max_y + pad)
                        if (max_y - min_y + 1) < MIN_DIMENSION:
                            min_y = builtins_max(0, max_y - MIN_DIMENSION + 1)

                    cropped_img = img_rgb.crop((min_x, min_y, max_x + 1, max_y + 1))
                    width, height = cropped_img.size
                    area = width * height

                    if area > MAX_AREA:
                        print("too large")
                        rejected_too_large.append({
                            'filename': os.path.basename(file),
                            'resistor_value_folder': os.path.basename(folder_root),
                            'filepath': image_path,
                            'crop_box': (min_x, min_y, max_x + 1, max_y + 1),
                            'width': width,
                            'height': height,
                            'area': area
                        })
                        continue

                    orig_w = matching_coords[:, 1].ptp() + 1
                    orig_h = matching_coords[:, 0].ptp() + 1
                    if width != orig_w or height != orig_h:
                        padded_count += 1

                    all_cropped_images_filtered.append({
                        'filename': os.path.basename(file),
                        'resistor_value_folder': os.path.basename(folder_root),
                        'cropped_image': cropped_img
                    })

            except Exception as e:
                print(f"Error processing {image_path}: {e}")

print(f"Processed:              {processed_image_count} images")
print(f"No color match:         {len(rejected_no_color)}")
print(f"Too large (>{MAX_AREA:,.0f} px): {len(rejected_too_large)}")
print(f"Padded to {MIN_DIMENSION}px min:     {padded_count}")
print(f"Kept:                   {len(all_cropped_images_filtered)} images")

In [ ]:
def display_rejects(reject_list, title, extra_info_fn=None):
    if not reject_list:
        print(f"No images in category: {title}")
        return

    print(f"\n--- {title} ({len(reject_list)} images) ---")
    num_images = len(reject_list)
    cols = 8
    rows = (num_images + cols - 1) // cols
    plt.figure(figsize=(2.5 * cols, 2.5 * rows))

    for i, data in enumerate(reject_list):
        # Load from filepath if no image object stored
        img = data.get('image') or data.get('cropped_image')
        if img is None and 'filepath' in data:
            img = Image.open(data['filepath']).convert('RGB')

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        label = f"{data['resistor_value_folder']}\n{data['filename']}"
        if extra_info_fn:
            label += f"\n{extra_info_fn(data)}"
        plt.title(label, fontsize=7)
        plt.axis('off')

    plt.suptitle(title, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

# Display no-color-match rejects (show full original image)
display_rejects(
    rejected_no_color,
    "Rejected: No Color Match"
)

# Display too-large rejects (show the crop, with its dimensions)
display_rejects(
    rejected_too_large,
    "Rejected: Too Large",
    extra_info_fn=lambda d: f"{d['width']}x{d['height']} | {d['area']:,} px"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

for data in rejected_too_large:
    full_img = Image.open(data['filepath']).convert('RGB')
    img = full_img.crop(data['crop_box'])
    img_array = np.array(img).astype(float)

    R = img_array[:, :, 0]
    G = img_array[:, :, 1]
    B = img_array[:, :, 2]

    safe_B = np.where(B > 0, B, 1)
    warmth_mask = (R / safe_B > WARMTH_RATIO) & (B > 0)
    brightness_mask = (R + G + B) >= MIN_BRIGHTNESS

    color_mask = np.zeros(R.shape, dtype=bool)
    for (r0, r1, g0, g1, b0, b1) in color_ranges:
        color_mask |= (
            (R >= r0) & (R <= r1) &
            (G >= g0) & (G <= g1) &
            (B >= b0) & (B <= b1)
        )

    combined_mask = color_mask & warmth_mask & brightness_mask
    combined_mask = apply_density_filter(combined_mask, DENSITY_KERNEL, DENSITY_THRESH)
    matching_coords = np.argwhere(combined_mask)

    if matching_coords.size == 0:
        continue

    extremes = {
        'top':    matching_coords[matching_coords[:, 0].argmin()],
        'bottom': matching_coords[matching_coords[:, 0].argmax()],
        'left':   matching_coords[matching_coords[:, 1].argmin()],
        'right':  matching_coords[matching_coords[:, 1].argmax()],
    }

    range_names = ['dark brown', 'mid brown', 'light brown']
    edge_colors = {'top': 'cyan', 'bottom': 'lime', 'left': 'red', 'right': 'yellow'}

    fig = plt.figure(figsize=(18, 5))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1])

    # Unmarked
    ax_clean = fig.add_subplot(gs[0])
    ax_clean.imshow(img)
    ax_clean.set_title("Unmarked", fontsize=9)
    ax_clean.axis('off')

    # Marked
    ax_img = fig.add_subplot(gs[1])
    ax_img.imshow(img)
    for edge, (py, px) in extremes.items():
        ax_img.plot(px, py, 'o', color=edge_colors[edge], markersize=10,
                    markeredgecolor='black', markeredgewidth=0.5, label=edge)
    ax_img.legend(fontsize=8, loc='upper right',
                  facecolor='black', labelcolor='white', framealpha=0.6)
    ax_img.set_title(f"{data['resistor_value_folder']} / {data['filename']}\n"
                     f"{data['width']}x{data['height']} | {data['area']:,} px", fontsize=9)
    ax_img.axis('off')

    # Table
    ax_tbl = fig.add_subplot(gs[2])
    ax_tbl.axis('off')

    col_labels = ['Edge', 'Coord (y, x)', 'R', 'G', 'B', 'Range']
    rows = []

    for edge, (py, px) in extremes.items():
        r = int(img_array[py, px, 0])
        g = int(img_array[py, px, 1])
        b = int(img_array[py, px, 2])
        matched = [range_names[i] for i, (r0,r1,g0,g1,b0,b1) in enumerate(color_ranges)
                   if r0<=r<=r1 and g0<=g<=g1 and b0<=b<=b1]
        rows.append([edge, f"({py}, {px})", str(r), str(g), str(b), ', '.join(matched) or '?'])

    table = ax_tbl.table(
        cellText=rows,
        colLabels=col_labels,
        cellLoc='center',
        loc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)

    for j in range(len(col_labels)):
        table[0, j].set_facecolor('#dddddd')
        table[0, j].set_text_props(fontweight='bold')

    for i in range(1, len(rows) + 1):
        table[i, 0].set_facecolor(edge_colors[rows[i-1][0]])
        table[i, 0].set_text_props(color='black', fontweight='bold')

    plt.suptitle(f"{data['resistor_value_folder']} / {data['filename']}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
def display_grid(image_list, title, extra_info_fn=None):
    if not image_list:
        print(f"No images in: {title}")
        return
    num_images = len(image_list)
    cols = 8
    rows = (num_images + cols - 1) // cols
    plt.figure(figsize=(2.5 * cols, 2.5 * rows))
    for i, data in enumerate(image_list):
        img = data.get('cropped_image') or data.get('image')
        w, h = img.size
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        label = f"{data['resistor_value_folder']}\n{data['filename']}"
        if extra_info_fn:
            label += f"\n{extra_info_fn(data)}"
        plt.title(label, fontsize=7)
        plt.axis('off')
    plt.suptitle(title, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

# Filter kept images that were padded
padded_images = []
for data in all_cropped_images_filtered:
    w, h = data['cropped_image'].size
    if w == MIN_DIMENSION or h == MIN_DIMENSION:
        padded_images.append(data)

display_grid(
    padded_images,
    f"Padded images (at least one dimension == {MIN_DIMENSION}px)",
    extra_info_fn=lambda d: f"{d['cropped_image'].size[0]}x{d['cropped_image'].size[1]}"
)

In [ ]:
display_grid(
    all_cropped_images_filtered,
    f"Padded images (at least one dimension == {MIN_DIMENSION}px)",
)

In [ ]:
import os
import csv
from PIL import Image

SAVE_DIR = 'cropped_output'

# Create output folders
os.makedirs(f'{SAVE_DIR}/kept',        exist_ok=True)
os.makedirs(f'{SAVE_DIR}/no_match',    exist_ok=True)
os.makedirs(f'{SAVE_DIR}/too_large',   exist_ok=True)

def save_with_csv(image_list, folder, get_image_fn, extra_fields_fn=None):
    csv_path = f'{SAVE_DIR}/{folder}/index.csv'
    with open(csv_path, 'w', newline='') as csvfile:
        fieldnames = ['saved_filename', 'original_filename', 'resistor_value_folder'] + \
                     (list(extra_fields_fn(image_list[0]).keys()) if extra_fields_fn and image_list else [])
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for i, data in enumerate(image_list):
            # Build a unique filename
            saved_name = f"{data['resistor_value_folder']}_{data['filename']}"
            saved_name = saved_name.replace(' ', '_').replace('/', '_')
            save_path = f'{SAVE_DIR}/{folder}/{saved_name}'

            # Save image
            img = get_image_fn(data)
            if img is not None:
                img.save(save_path)

            # Write CSV row
            row = {
                'saved_filename':         saved_name,
                'original_filename':      data['filename'],
                'resistor_value_folder':  data['resistor_value_folder'],
            }
            if extra_fields_fn:
                row.update(extra_fields_fn(data))
            writer.writerow(row)

    print(f"Saved {len(image_list)} images + index.csv → {SAVE_DIR}/{folder}/")

# --- Kept images ---
save_with_csv(
    all_cropped_images_filtered,
    'kept',
    get_image_fn=lambda d: d['cropped_image']
)

# --- No color match (load from filepath) ---
save_with_csv(
    rejected_no_color,
    'no_match',
    get_image_fn=lambda d: Image.open(d['filepath']).convert('RGB') if 'filepath' in d else d.get('image')
)

# --- Too large ---
save_with_csv(
    rejected_too_large,
    'too_large',
    get_image_fn=lambda d: d.get('image') or d.get('cropped_image'),
    extra_fields_fn=lambda d: {
        'width':  d['width'],
        'height': d['height'],
        'area':   d['area']
    }
)

In [ ]:
display_grid(
    all_cropped_images_filtered,
    "Kept images",
    extra_info_fn=lambda d: f"{d['cropped_image'].size[0]}x{d['cropped_image'].size[1]}"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# --- Histogram parameters ---
N_BINS = 8  # bins per channel, 8x8x8 = 512 total features

def compute_color_histogram(img, n_bins=N_BINS):
    """Compute a normalized RGB histogram for a PIL image."""
    img_array = np.array(img)
    
    r = img_array[:, :, 0].flatten()
    g = img_array[:, :, 1].flatten()
    b = img_array[:, :, 2].flatten()
    
    # Compute histogram for each channel
    r_hist, _ = np.histogram(r, bins=n_bins, range=(0, 256))
    g_hist, _ = np.histogram(g, bins=n_bins, range=(0, 256))
    b_hist, _ = np.histogram(b, bins=n_bins, range=(0, 256))
    
    # Normalize so values sum to 1
    total_pixels = len(r)
    r_hist = r_hist / total_pixels
    g_hist = g_hist / total_pixels
    b_hist = b_hist / total_pixels
    
    # Concatenate into single feature vector
    feature_vector = np.concatenate([r_hist, g_hist, b_hist])
    return feature_vector

def resize_and_histogram(data, target_size=(224, 224), n_bins=N_BINS):
    """Resize a cropped image and compute its histogram."""
    img = data['cropped_image']
    img_resized = img.resize(target_size, Image.LANCZOS)
    histogram = compute_color_histogram(img_resized, n_bins)
    return img_resized, histogram

# --- Process all kept images ---
processed_histograms = []

for data in all_cropped_images_filtered:
    img_resized, histogram = resize_and_histogram(data)
    processed_histograms.append({
        'filename':              data['filename'],
        'resistor_value_folder': data['resistor_value_folder'],
        'resized_image':         img_resized,
        'histogram':             histogram,
    })

print(f"Processed {len(processed_histograms)} histograms")
print(f"Feature vector size: {processed_histograms[0]['histogram'].shape[0]} values")

# --- Visualize a few examples ---
n_examples = builtins_min(5, len(processed_histograms))
fig, axes = plt.subplots(n_examples, 2, figsize=(10, 3 * n_examples))

bin_edges = [f"{int(i*256/N_BINS)}–{int((i+1)*256/N_BINS)}" for i in range(N_BINS)]
x = np.arange(N_BINS)
width = 0.25

for i in range(n_examples):
    data = processed_histograms[i]
    hist = data['histogram']
    r_hist = hist[0:N_BINS]
    g_hist = hist[N_BINS:2*N_BINS]
    b_hist = hist[2*N_BINS:3*N_BINS]

    # Image
    axes[i, 0].imshow(data['resized_image'])
    axes[i, 0].set_title(f"{data['resistor_value_folder']}\n{data['filename']}", fontsize=8)
    axes[i, 0].axis('off')

    # Histogram
    axes[i, 1].bar(x - width, r_hist, width, color='red',   alpha=0.7, label='R')
    axes[i, 1].bar(x,         g_hist, width, color='green', alpha=0.7, label='G')
    axes[i, 1].bar(x + width, b_hist, width, color='blue',  alpha=0.7, label='B')
    axes[i, 1].set_title('Color histogram', fontsize=8)
    axes[i, 1].set_xticks(x)
    axes[i, 1].set_xticklabels(bin_edges, rotation=45, fontsize=6)
    axes[i, 1].set_ylabel('Proportion', fontsize=8)
    axes[i, 1].legend(fontsize=7)

plt.suptitle(f'224x224 resized images + {N_BINS}-bin RGB histograms', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- Display all histograms ---
num_images = len(processed_histograms)
cols = 4  # pairs of (image, histogram)
rows = (num_images + cols - 1) // cols

fig, axes = plt.subplots(rows * 2, cols, figsize=(5 * cols, 4 * rows))

for i, data in enumerate(processed_histograms):
    row = (i // cols) * 2
    col = i % cols

    hist = data['histogram']
    r_hist = hist[0:N_BINS]
    g_hist = hist[N_BINS:2*N_BINS]
    b_hist = hist[2*N_BINS:3*N_BINS]

    # Image
    axes[row, col].imshow(data['resized_image'])
    axes[row, col].set_title(f"{data['resistor_value_folder']}\n{data['filename']}", fontsize=6)
    axes[row, col].axis('off')

    # Histogram
    x = np.arange(N_BINS)
    width = 0.25
    axes[row+1, col].bar(x - width, r_hist, width, color='red',   alpha=0.7, label='R')
    axes[row+1, col].bar(x,         g_hist, width, color='green', alpha=0.7, label='G')
    axes[row+1, col].bar(x + width, b_hist, width, color='blue',  alpha=0.7, label='B')
    axes[row+1, col].set_xticks(x)
    axes[row+1, col].set_xticklabels(
        [f"{int(j*256/N_BINS)}" for j in range(N_BINS)],
        rotation=45, fontsize=5
    )
    axes[row+1, col].set_ylabel('Proportion', fontsize=6)
    axes[row+1, col].legend(fontsize=5)

# Hide any unused subplots
for i in range(num_images, rows * cols):
    row = (i // cols) * 2
    col = i % cols
    axes[row, col].axis('off')
    axes[row+1, col].axis('off')

plt.suptitle('All images — 224x224 + RGB histograms', fontsize=12)
plt.tight_layout()
plt.show()

# --- Average histogram per class ---
from collections import defaultdict

class_histograms = defaultdict(list)
for data in processed_histograms:
    class_histograms[data['resistor_value_folder']].append(data['histogram'])

classes = sorted(class_histograms.keys())
n_classes = len(classes)
fig, axes = plt.subplots(1, n_classes, figsize=(5 * n_classes, 4), sharey=True)
if n_classes == 1:
    axes = [axes]

x = np.arange(N_BINS)
width = 0.25

for ax, cls in zip(axes, classes):
    hists = np.array(class_histograms[cls])
    avg = hists.mean(axis=0)
    std = hists.std(axis=0)

    r_avg, g_avg, b_avg = avg[0:N_BINS], avg[N_BINS:2*N_BINS], avg[2*N_BINS:3*N_BINS]
    r_std, g_std, b_std = std[0:N_BINS], std[N_BINS:2*N_BINS], std[2*N_BINS:3*N_BINS]

    ax.bar(x - width, r_avg, width, color='red',   alpha=0.7, label='R', yerr=r_std, capsize=3)
    ax.bar(x,         g_avg, width, color='green', alpha=0.7, label='G', yerr=g_std, capsize=3)
    ax.bar(x + width, b_avg, width, color='blue',  alpha=0.7, label='B', yerr=b_std, capsize=3)

    ax.set_title(f"{cls}\n({len(hists)} images)", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{int(j*256/N_BINS)}" for j in range(N_BINS)], rotation=45, fontsize=7)
    ax.set_ylabel('Proportion', fontsize=8)
    ax.legend(fontsize=7)

plt.suptitle('Average RGB histogram per resistor class (error bars = std dev)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import numpy as np
import matplotlib.pyplot as plt

# --- Prepare data ---
X = np.array([d['histogram'] for d in processed_histograms])
labels = np.array([d['resistor_value_folder'] for d in processed_histograms])
classes = sorted(set(labels))

# Assign a color to each class
cmap = plt.get_cmap('tab20')
class_colors = {cls: cmap(i / len(classes)) for i, cls in enumerate(classes)}

# --- PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# PCA plot
ax = axes[0]
for cls in classes:
    mask = labels == cls
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=cls, color=class_colors[cls],
               alpha=0.7, s=40, edgecolors='none')
ax.set_title(f'PCA — variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%', fontsize=10)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(fontsize=6, bbox_to_anchor=(1.01, 1), loc='upper left', ncol=1)
ax.grid(True, alpha=0.2)

# --- t-SNE ---
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X)

ax = axes[1]
for cls in classes:
    mask = labels == cls
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               label=cls, color=class_colors[cls],
               alpha=0.7, s=40, edgecolors='none')
ax.set_title('t-SNE', fontsize=10)
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
ax.legend(fontsize=6, bbox_to_anchor=(1.01, 1), loc='upper left', ncol=1)
ax.grid(True, alpha=0.2)

plt.suptitle('Histogram feature space — PCA vs t-SNE\nWell separated clusters = good for classification', fontsize=11)
plt.tight_layout()
plt.show()

# --- Quick separability check ---
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

le = LabelEncoder()
y = le.fit_transform(labels)
knn = KNeighborsClassifier(n_neighbors=5)
scores = cross_val_score(knn, X, y, cv=5)
print(f"5-fold cross-validation accuracy (KNN): {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%")
print(f"Chance level: {100/len(classes):.1f}%")

In [ ]:
# --- Filter out brown/black resistors ---
brown_black_classes = ['1 ohm', '100 ohm', '1K ohm', '10K ohm', '1M ohm', '1.2M ohms']

filtered_histograms = [d for d in processed_histograms 
                       if d['resistor_value_folder'] not in brown_black_classes]

print(f"Total images: {len(processed_histograms)}")
print(f"Filtered (non brown/black): {len(filtered_histograms)}")
print(f"Classes included: {sorted(set(d['resistor_value_folder'] for d in filtered_histograms))}")

# --- PCA + t-SNE ---
X_f = np.array([d['histogram'] for d in filtered_histograms])
labels_f = np.array([d['resistor_value_folder'] for d in filtered_histograms])
classes_f = sorted(set(labels_f))

cmap = plt.get_cmap('tab20')
class_colors = {cls: cmap(i / len(classes_f)) for i, cls in enumerate(classes_f)}

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_f)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_f)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, X_plot, title in zip(axes,
    [X_pca, X_tsne],
    [f'PCA — variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%', 't-SNE']):
    for cls in classes_f:
        mask = labels_f == cls
        ax.scatter(X_plot[mask, 0], X_plot[mask, 1],
                   label=cls, color=class_colors[cls],
                   alpha=0.7, s=40, edgecolors='none')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.grid(True, alpha=0.2)

plt.suptitle('Non brown/black resistors only — PCA vs t-SNE', fontsize=11)
plt.tight_layout()
plt.show()

# --- KNN accuracy ---
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

le = LabelEncoder()
y_f = le.fit_transform(labels_f)
knn = KNeighborsClassifier(n_neighbors=5)
scores = cross_val_score(knn, X_f, y_f, cv=5)

print(f"\n5-fold cross-validation accuracy (KNN): {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%")
print(f"Chance level: {100/len(classes_f):.1f}%")

In [ ]:
import os
import random
from PIL import Image

EXPORT_DIR = 'edge_impulse_dataset'
TRAIN_SPLIT = 0.8
TARGET_SIZE = (224, 224)

# Group images by class
from collections import defaultdict
class_images = defaultdict(list)
for data in all_cropped_images_filtered:
    class_images[data['resistor_value_folder']].append(data)

# Create folders and save
total_train, total_test = 0, 0

for cls, images in class_images.items():
    # Sanitize folder name
    cls_clean = cls.replace(' ', '_').replace('/', '_')
    
    # Shuffle and split
    random.shuffle(images)
    split_idx = int(len(images) * TRAIN_SPLIT)
    train_images = images[:split_idx]
    test_images  = images[split_idx:]

    for subset, subset_images in [('train', train_images), ('test', test_images)]:
        out_dir = os.path.join(EXPORT_DIR, subset, cls_clean)
        os.makedirs(out_dir, exist_ok=True)

        for data in subset_images:
            img_resized = data['cropped_image'].resize(TARGET_SIZE, Image.LANCZOS)
            save_name = f"{cls_clean}_{data['filename']}"
            img_resized.save(os.path.join(out_dir, save_name))

    total_train += len(train_images)
    total_test  += len(test_images)
    print(f"{cls:<20} train: {len(train_images):>3}  test: {len(test_images):>3}")

print(f"\nTotal train: {total_train}")
print(f"Total test:  {total_test}")
print(f"Saved to:    {EXPORT_DIR}/")

In [ ]:
import shutil
shutil.make_archive('edge_impulse_dataset', 'zip', EXPORT_DIR)
print("Saved edge_impulse_dataset.zip")

In [ ]:
import os
import random
import shutil
from PIL import Image
from collections import defaultdict

EXPORT_DIR = 'edge_impulse_dataset_v2'
TARGET_SIZE = (96, 96)  # match Edge Impulse project settings

# Clean up any previous export
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

# Group images by class
class_images = defaultdict(list)
for data in all_cropped_images_filtered:
    class_images[data['resistor_value_folder']].append(data)

# Save all images into flat per-class folders (no train/test split)
for cls, images in class_images.items():
    cls_clean = cls.replace(' ', '_').replace('/', '_')
    out_dir = os.path.join(EXPORT_DIR, cls_clean)
    os.makedirs(out_dir, exist_ok=True)

    for data in images:
        img_resized = data['cropped_image'].resize(TARGET_SIZE, Image.LANCZOS)
        save_name = f"{cls_clean}.{data['filename']}"
        img_resized.save(os.path.join(out_dir, save_name))

    print(f"{cls:<20} {len(images):>3} images")

# Zip it up
shutil.make_archive(EXPORT_DIR, 'zip', EXPORT_DIR)
print(f"\nSaved {EXPORT_DIR}.zip")